In [1]:
import pandas as pd 
import logging 
import duckdb 

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    filename='dataCleaning.log'
)
logger = logging.getLogger(__name__)

In [2]:
### load duckdb tables into pandas dfs 
con = None 
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.") 

    # inserting tables 
    cville = con.execute(f"""
        SELECT * FROM cville;
    """).fetchdf()
    dulles = con.execute(f"""
        SELECT * FROM dulles;
    """).fetchdf()
    lynchburg = con.execute(f"""
        SELECT * FROM lynchburg;
    """).fetchdf()
    norfolk = con.execute(f"""
        SELECT * FROM norfolk;
    """).fetchdf()
    dom = con.execute(f"""
        SELECT * FROM dom;
    """).fetchdf()

    logger.info("All tables loaded into pandas dataframes")

except Exception as e:
    logger.error(f"An error occurred: {e}")

finally:
    if con:
        con.close()
        logger.info("Duckdb connection closed.")

In [4]:
### combine all weather dfs for easier cleaning 
weatherdf = pd.concat([cville, dulles, lynchburg, norfolk], ignore_index=True)
weatherdf.drop(columns=['Entry_ID', 'REPORT_TYPE', 'SOURCE'], inplace=True)
weatherdf.head() 

,STATION,DATE,DailyAverageDewPointTemperature,DailyAverageDryBulbTemperature,DailyAverageRelativeHumidity,DailyAverageSeaLevelPressure,DailyAverageStationPressure,DailyAverageWetBulbTemperature,DailyAverageWindSpeed,DailyCoolingDegreeDays,...,ShortDurationPrecipitationValue030,ShortDurationPrecipitationValue045,ShortDurationPrecipitationValue060,ShortDurationPrecipitationValue080,ShortDurationPrecipitationValue100,ShortDurationPrecipitationValue120,ShortDurationPrecipitationValue150,ShortDurationPrecipitationValue180,Sunrise,Sunset
0,Charlottesville,2005-12-31 23:00:00,None,45,None,None,29.10,None,4.2,0,...,None,None,None,None,None,None,None,None,730.0,1704.0
1,Charlottesville,2006-01-01 23:00:00,None,42,None,None,29.36,None,1.8,0,...,None,None,None,None,None,None,None,None,730.0,1705.0
2,Charlottesville,2006-01-02 23:00:00,None,35,None,None,29.28,None,1.2,0,...,None,None,None,None,None,None,None,None,731.0,1705.0
3,Charlottesville,2006-01-03 23:00:00,None,44,None,None,29.11,None,1.7,0,...,None,None,None,None,None,None,None,None,731.0,1706.0
4,Charlottesville,2006-01-04 23:00:00,None,42,None,None,29.25,None,3.9,0,...,None,None,None,None,None,None,None,None,731.0,1707.0


In [13]:
### Drop unwanted columns && clean up values 
# list of non-daily columns to keep
cols_keep = ['STATION', 'DATE', 'Sunrise', 'Sunset']

# create list of columns to keep by adding daily columns to cols_keep
cols_keep2 = cols_keep + [col for col in weatherdf.columns if 'daily' in col.lower()] 

# new df of only keeping columns
weatherdf2 = weatherdf[cols_keep2].copy() 

# list of columns to clean
cols_fixing = ['DailyAverageDewPointTemperature', 'DailyAverageDryBulbTemperature',
       'DailyAverageRelativeHumidity', 'DailyAverageSeaLevelPressure',
       'DailyAverageStationPressure', 'DailyAverageWetBulbTemperature',
       'DailyAverageWindSpeed', 'DailyCoolingDegreeDays',
       'DailyDepartureFromNormalAverageTemperature', 'DailyHeatingDegreeDays',
       'DailyMaximumDryBulbTemperature', 'DailyMinimumDryBulbTemperature',
       'DailyPeakWindDirection', 'DailySustainedWindDirection', 'DailySustainedWindSpeed', 
       'DailyPeakWindSpeed', 'DailyPrecipitation', 'DailySnowfall', 'DailySnowDepth' ]

# for each column 
for col in cols_fixing:
    # fill any NaN values with 0 (to be all numerical)
    weatherdf2[col] = weatherdf2[col].fillna(0)
    # remove 's' from values
    weatherdf2[col] = weatherdf2[col].astype(str).str.replace('s', '', regex=False)
    # replace 'T' with 0 because represents trace amounts, which will be interpreted as 0 for this analysis 
    weatherdf2[col] = weatherdf2[col].astype(str).str.strip().replace('T', 0)

    # try to convert column to float 
    try:
        weatherdf2[col] = weatherdf2[col].astype(float)
        # print(col)
    # if fails, use pd.to_numeric with coercion
    except Exception as e:
        weatherdf2[col] = pd.to_numeric(weatherdf2[col], errors='coerce')
        # print(f"Column '{col}' had conversion issues, used pd.to_numeric as fallback.")

# ensure Date is in proper datetime format 
weatherdf2['DATE'] = pd.to_datetime(weatherdf2['DATE'], format='%Y-%m-%d') 
# extract month, day of week, and day of year for easier analysis later on
weatherdf2['Month'] = weatherdf2['DATE'].dt.month
weatherdf2['DayOfWeek'] = weatherdf2['DATE'].dt.dayofweek
weatherdf2['DayOfYear'] = weatherdf2['DATE'].dt.dayofyear


logger.info("Columns cleaned.")

# print columns to verify changes
weatherdf2.columns

Index(['STATION', 'DATE', 'Sunrise', 'Sunset',
       'DailyAverageDewPointTemperature', 'DailyAverageDryBulbTemperature',
       'DailyAverageRelativeHumidity', 'DailyAverageSeaLevelPressure',
       'DailyAverageStationPressure', 'DailyAverageWetBulbTemperature',
       'DailyAverageWindSpeed', 'DailyCoolingDegreeDays',
       'DailyDepartureFromNormalAverageTemperature', 'DailyHeatingDegreeDays',
       'DailyMaximumDryBulbTemperature', 'DailyMinimumDryBulbTemperature',
       'DailyPeakWindDirection', 'DailyPeakWindSpeed', 'DailyPrecipitation',
       'DailySnowDepth', 'DailySnowfall', 'DailySustainedWindDirection',
       'DailySustainedWindSpeed', 'DailyWeather', 'Month', 'DayOfWeek',
       'DayOfYear'],
      dtype='object')

In [14]:
# Classify extreme weather events (EWE) based on previous research done 

# make list of extreme weather codes to check for in the DailyWeather column 
extreme_codes = ['PL', 'DU', 'HZ', 'BLSN', 'FC', 'WIND', 'SN']

# iterate through each row and check for extreme weather conditions
for index, row in weatherdf2.iterrows():
    # convert to string to check for codes 
    weather = str(row['DailyWeather'])  
    if any(code in weather for code in extreme_codes):
        weatherdf2.at[index, 'EWE'] = 1

    # too high or too low pressure 
    elif row['DailyAverageStationPressure'] > 30.40 or row['DailyAverageStationPressure'] < 29.40:
        weatherdf2.at[index, 'EWE'] = 1

    # higher than 95 is extreme temp 
    elif row['DailyMaximumDryBulbTemperature'] > 95:
        weatherdf2.at[index, 'EWE'] = 1

    # lower than 10 is extreme temp
    elif row['DailyMinimumDryBulbTemperature'] < 10:
        weatherdf2.at[index, 'EWE'] = 1

    # daily precipitation greater than 2 inches is extreme
    elif row['DailyPrecipitation'] > 2:
        weatherdf2.at[index, 'EWE'] = 1
    
    # more than 2 inches of snow daily is extreme
    elif row['DailySnowfall'] > 2: 
        weatherdf2.at[index, 'EWE'] = 1
    
    # sustained wind speed greater than 58 mph is extreme
    elif row['DailySustainedWindSpeed'] > 58:
        weatherdf2.at[index, 'EWE'] = 1

    # else no extreme weather event
    else:
        weatherdf2.at[index, 'EWE'] = 0

logger.info("Extreme weather events classified.")

weatherdf2['EWE'].value_counts()

0.0    9547
1.0    9424
Name: EWE, dtype: int64

In [17]:
### combine weatherdf with energy df for easier analysis

# ensure both Date columns are in proper format before merging 
weatherdf2['DATE'] = pd.to_datetime(weatherdf2['DATE'])
dom['Datetime'] = pd.to_datetime(dom['Datetime'])
# round to the hour to prevent errors
dom['Datetime'] = dom['Datetime'].dt.floor('H')

# merge weather and energy dfs 
combined_df = pd.merge(weatherdf2, dom, left_on='DATE', right_on='Datetime', how='inner')
combined_df = combined_df.drop(columns=['Datetime'])
logger.info("Weather and energy dataframes merged into combined_df.")
combined_df  

,STATION,DATE,Sunrise,Sunset,DailyAverageDewPointTemperature,DailyAverageDryBulbTemperature,DailyAverageRelativeHumidity,DailyAverageSeaLevelPressure,DailyAverageStationPressure,DailyAverageWetBulbTemperature,...,DailySnowDepth,DailySnowfall,DailySustainedWindDirection,DailySustainedWindSpeed,DailyWeather,Month,DayOfWeek,DayOfYear,EWE,DOM_MW
0,Charlottesville,2005-12-31 23:00:00,730.0,1704.0,0.0,45.0,0.0,0.00,29.10,0.0,...,0.0,0.0,250.0,17.0,None,12,5,365,1.0,9991.0
1,Dulles,2005-12-31 23:00:00,728.0,1657.0,0.0,40.0,0.0,0.00,29.46,0.0,...,0.0,0.0,330.0,8.0,None,12,5,365,0.0,9991.0
2,Lynchburg,2005-12-31 23:00:00,731.0,1709.0,0.0,45.0,0.0,0.00,28.81,0.0,...,0.0,0.0,260.0,17.0,BR,12,5,365,1.0,9991.0
3,Norfolk,2005-12-31 23:00:00,718.0,1658.0,0.0,45.0,0.0,0.00,29.80,0.0,...,0.0,0.0,250.0,21.0,None,12,5,365,0.0,9991.0
4,Charlottesville,2006-01-01 23:00:00,730.0,1705.0,0.0,42.0,0.0,0.00,29.36,0.0,...,0.0,0.0,240.0,12.0,None,1,6,1,1.0,10467.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18362,Lynchburg,2016-11-12 23:00:00,654.0,1708.0,26.0,41.0,60.0,30.34,29.29,35.0,...,0.0,0.0,40.0,13.0,None,11,5,317,1.0,10825.0
18363,Norfolk,2016-11-12 23:00:00,641.0,1657.0,32.0,45.0,63.0,30.31,30.25,40.0,...,0.0,0.0,30.0,26.0,None,11,5,317,0.0,10825.0
18364,Dulles,2017-10-29 23:00:00,634.0,1713.0,54.0,55.0,96.0,29.49,29.20,54.0,...,0.0,0.0,300.0,28.0,RA MIFG BR FG,10,6,302,1.0,8868.0
18365,Lynchburg,2017-10-29 23:00:00,638.0,1722.0,48.0,51.0,84.0,29.49,28.53,50.0,...,0.0,0.0,170.0,15.0,RA BR,10,6,302,1.0,8868.0


In [18]:
### save combined_df as csv
combined_df.to_csv('data/combined_df.csv', index=False)

In [19]:
### save combined_df to duckdb table for easier querying later on
con = None
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.") 

    # inserting table 
    con.execute(f"""
        CREATE TABLE combined_df AS
        SELECT * FROM read_csv('data/combined_df.csv');
    """)

    logger.info("combined_df loaded into duckdb table.")

except Exception as e:
    logger.error(f"An error occurred: {e}")

finally:
    if con:
        con.close()
        logger.info("Duckdb connection closed.")